# 🧪 Praktikum 10 – RAG-Evaluation & Benchmarking
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** RAG-Systeme systematisch bewerten · Faithfulness (Faktentreue) vs. Relevanz unterscheiden · Ein automatisiertes Test-Skript entwickeln

⏱️ **Dauer:** 90 Minuten

In [5]:
import importlib
import os
import re
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
    "pandas": ("pandas", "3.0.2"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv ist erforderlich. Bitte installiere uv und fuehre die Zelle erneut aus.")

    commands = [
        ["uv", "pip", "install", "--python", sys.executable, *specs],
    ]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        commands = [command + ["--system"] for command in commands]

    last_error = None
    for command in commands:
        try:
            print("$", " ".join(command))
            subprocess.check_call(command)
            return
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Installation failed: {specs} ({last_error})")


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import ollama
import pandas as pd
import requests


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
subprocess.check_call(["ollama", "pull", MODEL], env=env)
payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
if MODEL not in available_models:
    raise RuntimeError(f"Missing local Ollama model: {MODEL}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Model:", MODEL)
print("Available local models:", ", ".join(available_models))

pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠸ pulling manifest ⠴ pulling manifest ⠦ 

Runtime: Local/Jupyter
Model: qwen3.5:0.8b
Available local models: qwen3.5:0.8b, nomic-embed-text:latest, devstral-small-2:latest, devstral-small-2:24b


pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling afb707b6b8fa: 100% ▕██████████████████▏ 1.0 GB                         
pulling 9be69ef46306: 100% ▕██████████████████▏  11 KB                         
pulling 9371364b27a5: 100% ▕██████████████████▏   65 B                         
pulling b14c6eab49f9: 100% ▕██████████████████▏  476 B                         
verifying sha256 digest 
writing manifest 
success 


## Teil 1 – Das Test-Set (Golden Dataset) ⏱️ 25 min
Wir erstellen eine Liste aus Fragen, Kontexten und Ground-Truth Antworten.

In [6]:
test_set = [
    {
        "q": "Wie hoch ist der Mount Everest?",
        "c": "Der Mount Everest ist 8848 Meter hoch.",
        "gt": "8848 Meter"
    },
    {
        "q": "Wer schrieb Faust?",
        "c": "Goethe verfasste das Drama Faust.",
        "gt": "Johann Wolfgang von Goethe"
    }
]

def get_answer(query, context):
    prompt = (
        "Beantworte die Frage ausschliesslich mit Informationen aus dem Kontext. "
        "Antworte kurz und praezise in einem Satz.\n\n"
        f"Kontext: {context}\nFrage: {query}"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 40, "stop": ["\n\n"]},
        keep_alive="10m",
    )
    return response["message"]["content"].strip()

results = []
for item in test_set:
    ans = get_answer(item['q'], item['c'])
    results.append({"query": item['q'], "answer": ans, "ground_truth": item['gt']})

df = pd.DataFrame(results)
print(df)

                             query  \
0  Wie hoch ist der Mount Everest?   
1               Wer schrieb Faust?   

                                              answer  \
0             Der Mount Everest ist 8848 Meter hoch.   
1  Johann Wolfgang von Goethe verfasste das Drama...   

                 ground_truth  
0                  8848 Meter  
1  Johann Wolfgang von Goethe  


## Teil 2 – Automated LLM-as-a-Judge ⏱️ 40 min
Wir lassen ein LLM prüfen, ob die `answer` zur `ground_truth` passt.

In [7]:
def extract_judge_score(text):
    matches = re.findall(r"\d+(?:\.\d+)?", text)
    if not matches:
        raise RuntimeError(f"Judge output does not contain a numeric score: {text!r}")

    score = float(matches[0])
    if not 0.0 <= score <= 1.0:
        raise RuntimeError(f"Judge score must be between 0.0 and 1.0, got {score}")
    return score


def judge_answer(query, answer, gt):
    prompt = f"""Bewerte die Korrektheit der Antwort basierend auf der Ground Truth.
    Frage: {query}
    Antwort: {answer}
    Ground Truth: {gt}

    Gib einen Score von 0 (falsch) bis 1 (korrekt). Antworte NUR mit der Zahl.
    """
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 8, "stop": ["\n"]},
        keep_alive="10m",
    )
    return extract_judge_score(response["message"]["content"])


df["score"] = df.apply(lambda row: judge_answer(row["query"], row["answer"], row["ground_truth"]), axis=1)
print(f"Durchschnittlicher Score: {df['score'].mean()}")

Durchschnittlicher Score: 1.0


## Teil 3 – Faithfulness Check ⏱️ 25 min
Halluziniert das Modell? Wir prüfen, ob die Antwort NUR Informationen aus dem Kontext enthält.

In [8]:
def check_faithfulness(context, answer):
    prompt = (
        "Enthaelt die Antwort Informationen, die NICHT im Kontext stehen? "
        "Antworte exakt mit JA oder NEIN.\n\n"
        f"Kontext: {context}\nAntwort: {answer}"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 8, "stop": ["\n"]},
        keep_alive="10m",
    )
    return response["message"]["content"].strip()

print("Faithfulness Check:", check_faithfulness(test_set[0]['c'], "Der Mount Everest ist 8848m hoch und liegt im Himalaya."))

Faithfulness Check: JA


## 🧩 Aufgaben
1. **Diverse Dataset:** Erweitere das `test_set` auf 10 verschiedene Fragen (auch Fangfragen!).
2. **Metric Comparison:** Berechne für deine Ergebnisse zusätzlich den Jaccard-Score. Wie stark korreliert er mit dem LLM-Judge?
3. **RAGAS:** Recherchiere das Framework 'RAGAS'. Welche 3 Hauptmetriken werden dort zur Bewertung von RAG genutzt?